In [ ]:
!nvidia-smi

Fri Sep 25 11:52:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    datasets \
    huggingface_hub \
    gradio \
    pillow

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
import torch

from transformers import (
    AutoConfig,
    AutoProcessor,
    BitsAndBytesConfig,
    LlavaForConditionalGeneration,
)

from PIL import Image

print("Imports successful.")

Imports successful.


In [ ]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

DEVICE = "cuda"
COMPUTE_DTYPE = torch.float16

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Compute dtype:", COMPUTE_DTYPE)

Model: llava-hf/llava-1.5-7b-hf
Device: cuda
Compute dtype: torch.float16


In [ ]:
config = AutoConfig.from_pretrained(MODEL_ID)

print("Vision image size:", config.vision_config.image_size)
print("Vision patch size:", config.vision_config.patch_size)

Vision image size: 336
Vision patch size: 14


In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    use_fast=False
)

processor.patch_size = config.vision_config.patch_size
processor.vision_feature_select_strategy = (
    config.vision_feature_select_strategy
)
processor.num_additional_image_tokens = 1

processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Processor loaded.")

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Processor loaded.


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print("4-bit quantization configured.")

4-bit quantization configured.


In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

print("LLaVA loaded successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

LLaVA loaded successfully.


In [ ]:
PROJECT_NAME = "LLaVA Visual Product Chatbot"

MAX_SEQUENCE_LENGTH = 1024
MAX_NEW_TOKENS = 64

print("Project:", PROJECT_NAME)

Project: LLaVA Visual Product Chatbot


In [ ]:
def build_training_messages(question, answer):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": question
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": answer
                }
            ],
        },
    ]

In [ ]:
def build_user_messages(question):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": question
                },
            ],
        }
    ]

In [ ]:
from datasets import load_dataset

DATASET_ID = "ashraq/fashion-product-images-small"

dataset = load_dataset(
    DATASET_ID,
    split="train"
)

print(dataset)

Dataset({
    features: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image'],
    num_rows: 44072
})


In [ ]:
QUESTION_TEMPLATES = {

    "category": [
        "What type of product is this?",
        "What kind of item is shown here?",
        "What product is shown in the image?",
        "Can you identify this product?",
        "What kind of product is this?"
    ],

    "color": [
        "What color is this product?",
        "Which color is this product?",
        "What color is the item?",
        "Which color is it?",
        "Can you tell me the color of this product?"
    ],

    "description": [
        "Describe this product briefly.",
        "Can you briefly describe this product?",
        "What do you see in this image?",
        "Can you describe the item shown?",
        "Give me a short description of this product."
    ],

    "category_color": [
        "What product is this and what color is it?",
        "What type of product is shown and what color is it?",
        "Can you identify the product and its color?",
        "What is this item and what color is it?",
        "Tell me the product type and its color."
    ],

    "brand_unknown": [
        "What brand is this product?",
        "Do you know the brand of this item?",
        "Who makes this product?",
        "Can you identify the brand?",
        "What company made this product?"
    ],

    "price_unknown": [
        "How much does this product cost?",
        "What is the price of this product?",
        "Can you tell me how much this costs?",
        "How much is this item?",
        "What does this product cost?"
    ],

    "material_unknown": [
        "What material is this product made from?",
        "What is this product made of?",
        "Can you tell what this item is made of?",
        "What material is this item?",
        "Can you identify the material?"
    ]
}

QUESTION_TYPES = list(QUESTION_TEMPLATES.keys())

def format_category(category):
    category = str(category).strip()

    # Dataset labels are often plural:
    # Wallets, Tshirts, Sports Shoes, Flip Flops...
    # Avoid incorrect constructions such as "a wallets".
    return category.lower()


def build_vqa_pair(example, question_type, template_index=0):

    category = format_category(
        example["articleType"]
    )

    color = str(
        example["baseColour"]
    ).strip().lower()

    templates = QUESTION_TEMPLATES[
        question_type
    ]

    question = templates[
        template_index % len(templates)
    ]

    if question_type == "category":

        answer = (
            f"The product category is {category}."
        )

    elif question_type == "color":

        answer = (
            f"The product is {color}."
        )

    elif question_type == "description":

        answer = (
            f"The image shows {color} {category}."
        )

    elif question_type == "category_color":

        answer = (
            f"The product category is {category} "
            f"and the color is {color}."
        )

    elif question_type == "brand_unknown":

        answer = (
            "The brand cannot be reliably "
            "determined from the image."
        )

    elif question_type == "price_unknown":

        answer = (
            "The price cannot be determined "
            "from the image."
        )

    elif question_type == "material_unknown":

        answer = (
            "The material cannot be reliably "
            "determined from the image."
        )

    else:
        raise ValueError(
            f"Unknown question type: {question_type}"
        )

    return question, answer

In [ ]:
def add_vqa_fields(example, idx):

    question_type_index = (
        idx % len(QUESTION_TYPES)
    )

    question_type = QUESTION_TYPES[
        question_type_index
    ]

    # Change wording across examples even when
    # the intent/question type is the same.
    template_index = (
        idx // len(QUESTION_TYPES)
    ) % 5

    question, answer = build_vqa_pair(
        example,
        question_type,
        template_index
    )

    return {
        "question_type": question_type,
        "template_index": template_index,
        "question": question,
        "answer": answer,
    }

In [ ]:
SEED = 42

dataset = dataset.shuffle(seed=SEED)

small_dataset = dataset.select(
    range(min(2000, len(dataset)))
)

print("Selected samples:", len(small_dataset))

Selected samples: 2000


In [ ]:
split_1 = small_dataset.train_test_split(
    test_size=0.2,
    seed=SEED
)

split_2 = split_1["test"].train_test_split(
    test_size=0.5,
    seed=SEED
)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1600
Validation: 200
Test: 200


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("PEFT imports successful.")

PEFT imports successful.


In [ ]:
train_dataset = train_dataset.map(
    add_vqa_fields,
    with_indices=True
)

val_dataset = val_dataset.map(
    add_vqa_fields,
    with_indices=True
)

test_dataset = test_dataset.map(
    add_vqa_fields,
    with_indices=True
)

print("VQA fields added.")

VQA fields added.


In [ ]:
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

print("LoRA target modules:")
print(LORA_TARGET_MODULES)

LoRA target modules:
['q_proj', 'k_proj', 'v_proj', 'o_proj']


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'q_proj', 'v_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [ ]:
class LlavaDataCollator:

    def __init__(self, processor, max_length=1024):
        self.processor = processor
        self.max_length = max_length

    def __call__(self, examples):

        images = []
        full_texts = []
        prompt_texts = []

        for example in examples:

            image = example["image"].convert("RGB")

            question = example["question"]
            answer = example["answer"]

            # USER + ASSISTANT
            full_messages = build_training_messages(
                question,
                answer
            )

            full_text = self.processor.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False
            )

            # USER only
            user_messages = build_user_messages(
                question
            )

            prompt_text = self.processor.apply_chat_template(
                user_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            images.append(image)
            full_texts.append(full_text)
            prompt_texts.append(prompt_text)

        batch = self.processor(
            images=images,
            text=full_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        prompt_batch = self.processor(
            images=images,
            text=prompt_texts,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors=None
        )

        labels = batch["input_ids"].clone()

        # Padding loss'a dahil edilmesin.
        labels[
            batch["attention_mask"] == 0
        ] = -100

        # User question loss'a dahil edilmesin.
        # Sadece assistant answer üzerinden training yap.
        for i, prompt_ids in enumerate(
            prompt_batch["input_ids"]
        ):
            prompt_length = len(prompt_ids)
            labels[i, :prompt_length] = -100

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = LlavaDataCollator(
    processor=processor,
    max_length=MAX_SEQUENCE_LENGTH
)

print("Data collator ready.")

Data collator ready.


In [ ]:
import gc
import torch

try:
    del trainer
except:
    pass

try:
    del fast_trainer
except:
    pass

try:
    del model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

CUDA allocated: 0.0 GB


In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

print("Fresh base LLaVA loaded.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Fresh base LLaVA loaded.


In [ ]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False
)

model.gradient_checkpointing_disable()
model.config.use_cache = False

print("Model prepared for QLoRA.")
print(
    "Gradient checkpointing:",
    model.is_gradient_checkpointing
)

Model prepared for QLoRA.
Gradient checkpointing: False


In [ ]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 19,136,512 || all params: 7,082,563,584 || trainable%: 0.2702


In [ ]:
FAST_TRAIN_SIZE = 300
FAST_VAL_SIZE = 50

fast_train_dataset = train_dataset.select(
    range(min(FAST_TRAIN_SIZE, len(train_dataset)))
)

fast_val_dataset = val_dataset.select(
    range(min(FAST_VAL_SIZE, len(val_dataset)))
)

print("Training samples:", len(fast_train_dataset))
print("Validation samples:", len(fast_val_dataset))

Training samples: 300
Validation samples: 50


In [ ]:
from transformers import TrainingArguments, Trainer

print(TrainingArguments)
print(Trainer)

<class 'transformers.training_args.TrainingArguments'>
<class 'transformers.trainer.Trainer'>


In [ ]:
CHATBOT_OUTPUT_DIR = "./llava-product-chatbot-v2-fast"

chatbot_training_args = TrainingArguments(
    output_dir=CHATBOT_OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    fp16=True,
    bf16=False,

    lr_scheduler_type="cosine",
    warmup_steps=2,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    gradient_checkpointing=False,

    report_to="none",
    seed=42,
)

In [ ]:
chatbot_trainer = Trainer(
    model=model,
    args=chatbot_training_args,

    train_dataset=fast_train_dataset,
    eval_dataset=fast_val_dataset,

    data_collator=data_collator,
)

print("Chatbot trainer ready.")

Chatbot trainer ready.


In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(
    "CUDA allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "CUDA reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

CUDA allocated: 4.36 GB
CUDA reserved: 4.88 GB


In [ ]:
chatbot_result = chatbot_trainer.train()

Step,Training Loss,Validation Loss
25,0.687262,0.298842
50,0.240330,0.151343
75,0.215897,0.133062


In [ ]:
print("Training metrics:")

for key, value in chatbot_result.metrics.items():
    print(f"{key}: {value}")

print(
    "\nPeak CUDA memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "\nBest checkpoint:",
    chatbot_trainer.state.best_model_checkpoint
)

print(
    "Best validation loss:",
    chatbot_trainer.state.best_metric
)

Training metrics:
train_runtime: 632.1917
train_samples_per_second: 0.475
train_steps_per_second: 0.119
total_flos: 7580630572701696.0
train_loss: 0.6384175475438436
epoch: 1.0

Peak CUDA memory: 10.88 GB

Best checkpoint: ./llava-product-chatbot-v2-fast/checkpoint-75
Best validation loss: 0.13306169211864471


In [ ]:
CHATBOT_ADAPTER_PATH = "./llava-product-chatbot-v2-lora"

model.save_pretrained(
    CHATBOT_ADAPTER_PATH
)

processor.save_pretrained(
    CHATBOT_ADAPTER_PATH
)

print(
    "Chatbot adapter saved to:",
    CHATBOT_ADAPTER_PATH
)

Chatbot adapter saved to: ./llava-product-chatbot-v2-lora


In [ ]:
model.eval()
model.config.use_cache = True

print("Model ready for inference.")

Model ready for inference.


In [ ]:
import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.28.0


In [ ]:
def generate_product_answer(image, question):

    if image is None:
        return "Please upload a product image first."

    if question is None or not question.strip():
        return "Please enter a question."

    model.eval()
    model.config.use_cache = True

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": question.strip()
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        images=image.convert("RGB"),
        text=prompt,
        return_tensors="pt"
    )

    inputs = {
        key: (
            value.to(
                device=DEVICE,
                dtype=COMPUTE_DTYPE
            )
            if value.is_floating_point()
            else value.to(DEVICE)
        )
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    answer = processor.batch_decode(
        output_ids[:, input_length:],
        skip_special_tokens=True
    )[0].strip()

    # Prevent the model from continuing with
    # additional generated questions/answers.
    answer = answer.split("\n")[0].strip()

    return answer

In [ ]:
def respond_to_product(
    image,
    message,
    history
):

    if history is None:
        history = []

    if image is None:
        return (
            history,
            "",
            "⚠️ Please upload a product image first."
        )

    if message is None or not message.strip():
        return (
            history,
            "",
            "⚠️ Please enter a question."
        )

    try:

        question = message.strip()

        answer = generate_product_answer(
            image,
            question
        )

        history = history + [
            {
                "role": "user",
                "content": question
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        return (
            history,
            "",
            "✅ Ready"
        )

    except Exception as e:

        return (
            history,
            message,
            f"❌ Error: {str(e)}"
        )

In [ ]:
def clear_product_chat():

    return (
        [],
        "",
        None,
        "Upload an image to start."
    )

In [ ]:
with gr.Blocks(
    title="LLaVA Visual Product Chatbot"
) as demo:

    gr.Markdown(
        """
        # 🛍️ LLaVA Visual Product Chatbot

        Upload a product image and ask questions about it.

        **Model:** LLaVA-1.5-7B
        **Fine-tuning:** QLoRA · 4-bit NF4

        The assistant can identify visible product attributes
        and avoids guessing information that cannot be reliably
        determined from the image.
        """
    )

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=1):

            product_image = gr.Image(
                type="pil",
                label="Product Image",
                height=420
            )

            status = gr.Markdown(
                "Upload an image to start."
            )

        # RIGHT SIDE
        with gr.Column(scale=2):

            chatbot = gr.Chatbot(
                label="Product Assistant",
                height=420
            )

            message = gr.Textbox(
                label="Ask about the product",
                placeholder=(
                    "e.g. What color is this product?"
                ),
                lines=1
            )

            with gr.Row():

                send_button = gr.Button(
                    "Ask",
                    variant="primary"
                )

                clear_button = gr.Button(
                    "Clear"
                )

    gr.Markdown(
        """
        ### Example questions

        - What type of product is this?
        - What color is it?
        - Describe this product briefly.
        - Do you know the brand?
        - How much does this product cost?
        - What material is this product made from?
        """
    )

    # Ask button
    send_button.click(
        fn=respond_to_product,
        inputs=[
            product_image,
            message,
            chatbot
        ],
        outputs=[
            chatbot,
            message,
            status
        ]
    )

    # Enter key
    message.submit(
        fn=respond_to_product,
        inputs=[
            product_image,
            message,
            chatbot
        ],
        outputs=[
            chatbot,
            message,
            status
        ]
    )

    # Clear
    clear_button.click(
        fn=clear_product_chat,
        inputs=[],
        outputs=[
            chatbot,
            message,
            product_image,
            status
        ]
    )

In [ ]:
demo.queue()

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e6ae49a82e678d3f4e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e6ae49a82e678d3f4e.gradio.live


In [3]:
# GitHub repository setup
GITHUB_USERNAME = "berkin9"
REPO_NAME = "LLaVA-Visual-Product-Chatbot"

!git config --global user.name "berkin9"
!git config --global user.email "berkinakbiyik@hotmail.com"

!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

%cd /content/{REPO_NAME}

Cloning into 'LLaVA-Visual-Product-Chatbot'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 6.34 KiB | 6.34 MiB/s, done.
/content/LLaVA-Visual-Product-Chatbot


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
!cp "/content/drive/MyDrive/Colab Notebooks/Llava + Lora.ipynb" \
"/content/LLaVA-Visual-Product-Chatbot/llava_visual_product_chatbot.ipynb"

In [7]:
!ls "/content/drive/MyDrive/Colab Notebooks" | tail -30

NER_using_CRF_and_BERT (1).ipynb
NER_using_CRF_and_BERT (2).ipynb
NER_using_CRF_and_BERT.ipynb
NLP Ass.ipynb
NLP Ass.ipynb adlı not defterinin kopyası
NLTK.ipynb
OpenVino.ipynb
Part_1__fine_tune_medical_blip.ipynb
Part_1_LLaVA_2026.ipynb
Part_1_LLM_Components_FIXED_extended.ipynb
Qwen2_5_Vision_Language_Model_(7B)_+_LoRA_Adapter_+_Unsloth.ipynb
Restaurant Chatbot.ipynb
Restaurant_Reservation_Assistant_A_Modular_Streamlit_Project_v2.ipynb
Stemming_and_Lemmatizing.ipynb
TravelCompanion_Colab_V2 (1).ipynb
TravelCompanion_Colab_V2.ipynb
Untitled
Untitled (1)
Untitled (2)
Version_3_Transcribe_Translate_with_OpenAI_Whisper.ipynb
Week10.ipynb
week2.ipynb
Week_2_Part1_VQA_Chatbot_updated_Last_version.ipynb
week3 4.ipynb
week4-RF.jpynb
week5.ipynb
Week6.ipynb
Week7.ipynb
week8.ipynb
YOLO.ipynb
